In [ ]:
# Install dependencies
!pip install -q timm pytorch-msssim lpips scikit-image thop gdown transformers accelerate

import os
import shutil
import subprocess
import json
from PIL import Image

# --- 1. MOUNTED ARTIFACT PATHS (UPDATE THESE) ---
MOUNTED_SATEHAZE_WEIGHTS_BASE = '/kaggle/input/notebooks/markosumire/nonvlm/weights/SateHaze1k_Baseline/model_best.pth'
MOUNTED_SATEHAZE_WEIGHTS_VLM  = '/kaggle/input/notebooks/markosumire/vlmtraining/weights/SateHaze1k_VLM/model_best.pth'
MOUNTED_CAPTIONS_BASE         = '/kaggle/input/notebooks/markosumire/onlycaptioningcommit/captions'

# --- 2. CLONE REPOSITORY ---
!git clone https://github.com/sumire25/RSHazeNet.git /kaggle/working/RSHazeNet
%cd /kaggle/working/RSHazeNet

# --- 3. SETUP RRSHID DATASET ---
RRSHID_ZIP = '/kaggle/working/RRSHID.zip'
RRSHID_BASE = '/kaggle/working/RRSHID'
if not os.path.isdir(RRSHID_BASE):
    !wget -q -O {RRSHID_ZIP} "https://github.com/lwCVer/RRSHID/releases/download/dataset/RRSHID.zip"
    !unzip -q {RRSHID_ZIP} -d {RRSHID_BASE}
    !rm -f {RRSHID_ZIP}

# --- 4. CROSS-DOMAIN DATASET AGGREGATOR (RESIZING LOGIC) ---
UNIFIED_CROSS_DOMAIN = '/kaggle/working/Unified_CrossDomain_RRSHID'

def prepare_cross_domain_layout(source_base, unified_base, levels, splits, original_base_for_key):
    if os.path.exists(unified_base): shutil.rmtree(unified_base)
    
    for split in splits:
        hazy_dir = os.path.join(unified_base, split, 'hazy')
        gt_dir = os.path.join(unified_base, split, 'GT')
        os.makedirs(hazy_dir, exist_ok=True)
        os.makedirs(gt_dir, exist_ok=True)
        
        master_captions = {}
        
        for level in levels:
            raw_split = os.path.join(source_base, level, split)
            if not os.path.isdir(raw_split): continue
            raw_hazy = os.path.join(raw_split, 'hazy')
            raw_gt = next((os.path.join(raw_split, c) for c in ['GT', 'gt', 'clear'] if os.path.isdir(os.path.join(raw_split, c))), None)
            
            if not raw_gt or not os.path.isdir(raw_hazy): continue
            
            for fname in os.listdir(raw_hazy):
                if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')): continue
                unique_fname = f"{level}_{fname}"
                
                # Resize Hazy input to 512x512 to match SateHaze1k's receptive field
                hazy_path = os.path.join(raw_hazy, fname)
                with Image.open(hazy_path) as img:
                    img_resized = img.resize((512, 512), Image.BICUBIC)
                    img_resized.save(os.path.join(hazy_dir, unique_fname))
                
                # Keep GT at native 256x256 using symlink
                os.symlink(os.path.join(raw_gt, fname), os.path.join(gt_dir, unique_fname))
                
            # Merge JSON metadata for VLM evaluation
            cap_key = f"{original_base_for_key}/{level}/{split}/hazy".replace('/', '_').strip('_')
            cap_path = os.path.join(MOUNTED_CAPTIONS_BASE, cap_key, 'captions.json')
            
            if os.path.exists(cap_path):
                with open(cap_path, 'r') as f:
                    level_caps = json.load(f)
                    for k, v in level_caps.items():
                        master_captions[f"{level}_{k}"] = v
        
        if master_captions:
            with open(os.path.join(hazy_dir, 'captions.json'), 'w') as f:
                json.dump(master_captions, f, indent=4)

print("Preparing Cross-Domain Dataset (Scaling hazy images to 512x512)...")
prepare_cross_domain_layout(RRSHID_BASE, UNIFIED_CROSS_DOMAIN, 
                            ['thin_fog', 'moderate_fog', 'thick_fog'], ['test'],
                            '/kaggle/working/RRSHID')

# --- 5. TEST ORCHESTRATOR ---
def run_cross_domain_test(model_name, unified_base, weights_path, use_vlm=False):
    mode_str = "VLM" if use_vlm else "Baseline"
    print(f"\n{'='*50}\nTesting Cross-Domain ({mode_str}): {model_name} on RRSHID\n{'='*50}")
    
    result_dir = f'/kaggle/working/results/CrossDomain_{model_name}_{mode_str}'
    os.makedirs(result_dir, exist_ok=True)
    
    # Inference Call
    test_cmd = [
        'python', 'infer.py',
        '--test_input', f'{unified_base}/test/hazy',
        '--test_target', f'{unified_base}/test/GT',
        '--weights', weights_path,
        '--result_path', result_dir
    ]
    if use_vlm:
        test_cmd.append('--caption')
        
    print(f"-> Running Inference ({mode_str})...")
    subprocess.run(test_cmd, check=True)
    
    # Metric Evaluation
    eval_cmd = [
        'python', 'evaluate.py',
        '--train_folder', result_dir,
        '--target_folder', f'{unified_base}/test/GT'
    ]
    print("-> Computing Evaluation Metrics...")
    
    metrics_file = os.path.join(result_dir, 'metrics.txt')
    with open(metrics_file, 'w') as f:
        subprocess.run(eval_cmd, check=True, stdout=f, text=True)
        
    with open(metrics_file, 'r') as f:
        print(f.read())

# --- 6. EXECUTE PIPELINE ---
run_cross_domain_test("SateHaze1k", UNIFIED_CROSS_DOMAIN, MOUNTED_SATEHAZE_WEIGHTS_BASE, use_vlm=False)
run_cross_domain_test("SateHaze1k", UNIFIED_CROSS_DOMAIN, MOUNTED_SATEHAZE_WEIGHTS_VLM, use_vlm=True)

print("\nCross-Domain Testing Complete! Outputs and metrics are available in /kaggle/working/results/")